# `web_search` provider playground

Tests every provider through the unified `web_search()` interface from `tools/web_search.py`.  
**Run the Setup cell first**, then each provider block independently.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
from dotenv import load_dotenv

# This notebook lives at mcp/server/tools/ — project root is 3 levels up
ROOT        = Path("__file__").resolve().parents[3]   # HSCodeSearch/
SERVER_ROOT = ROOT / "mcp" / "server"

for p in (str(ROOT), str(SERVER_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(ROOT / ".env")

from tools.web_search import (
    web_search,
    list_providers,
    get_current_config,
    get_available_providers,
)

QUERY = "HS code for Samsung Galaxy S8 smartphone"

def show(result: dict):
    print(f"provider : {result['provider']}")
    print(f"model    : {result.get('model', '')}")
    if result.get("answer"):
        print(f"answer   : {result['answer'][:400]}")
    hits = result.get("search_results", [])
    print(f"results  : {len(hits)}")
    for i, r in enumerate(hits[:5], 1):
        print(f"  [{i}] {r['title']}")
        print(f"      {r['url']}")
        if r.get("snippet"):
            print(f"      {r['snippet'][:110]}")

print("Setup OK  |  all providers:", list_providers())
print("Config    :", get_current_config()["provider"], "/ available:", get_available_providers())

In [ ]:
# ── 1. DuckDuckGo (no API key) ─────────────────────────────────────────────
result = web_search(QUERY, provider="duckduckgo")
show(result)

In [ ]:
# ── 2. Serper — Google SERP (SERPER_API_KEY) ───────────────────────────────
result = web_search(QUERY, provider="serper", num=5)
show(result)

In [ ]:
# ── 3. Tavily AI Search (TAVILY_API_KEY) ───────────────────────────────────
result = web_search(QUERY, provider="tavily", max_results=5)
show(result)

In [ ]:
# ── 4. Perplexity AI — LLM answer + citations (PERPLEXITY_API_KEY) ─────────
result = web_search(
    "What is the 6-digit HS code for Samsung Galaxy S8? Give chapter and heading.",
    provider="perplexity",
)
show(result)

In [ ]:
# ── 5. Jina AI Reader (JINA_API_KEY — set key to activate) ─────────────────
if os.getenv("JINA_API_KEY"):
    result = web_search(QUERY, provider="jina", num=5)
    show(result)
else:
    print("⚠️  JINA_API_KEY not set — skipping.")

In [ ]:
# ── 6. Brave Search (BRAVE_API_KEY — set key to activate) ──────────────────
if os.getenv("BRAVE_API_KEY"):
    result = web_search(QUERY, provider="brave", count=5)
    show(result)
else:
    print("⚠️  BRAVE_API_KEY not set — skipping.")

In [ ]:
# ── 7. SearXNG self-hosted (SEARXNG_BASE_URL — set to activate) ────────────
if os.getenv("SEARXNG_BASE_URL"):
    result = web_search(QUERY, provider="searxng")
    show(result)
else:
    print("⚠️  SEARXNG_BASE_URL not set — skipping.")

In [ ]:
# ── 8. WebCrawler — crawl a real HS-code lookup page ──────────────────────
import concurrent.futures

URL = "https://www.i5a6.com/hscode/key/%E7%83%98%E5%B9%B2%E6%9C%BA"

# Playwright sync API conflicts with Jupyter's event loop — run in a thread.
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as exe:
    result = exe.submit(web_search, URL, None, False, "webcrawler").result(timeout=60)

print(f"provider : {result['provider']}")
print(f"crawled  : {result['metadata'].get('crawl_success')}")
print(f"\n── First 800 chars ──")
print(result["answer"][:800])